# Human vs Machine Translation Detection Codebase

This Jupyter Notebook is designed to prepare a dataset for detecting human versus machine-generated translations. It leverages the SMOL dataset from Hugging Face, focusing on low-resource languages such as Amharic, Swahili, and Afrikaans.

## Workflow Overview

1. **Data Loading**: Load and process the first 200 rows from specified SMOL configurations, extracting human translations (English source) and target-language texts.

2. **CSV Generation**: Combine the loaded data into a single CSV file, preserving the order of languages.

3. **Integration of Machine Translations**: Add columns from text files containing machine-generated translations (e.g., from models like Gemini) to the CSV, ensuring row alignment.

4. **Output**: Produce an updated CSV with human and machine translations for further analysis or model training in translation detection tasks.

The notebook includes error handling for data mismatches and provides sanity checks on the final dataset. Ensure paths are correctly set in the relevant cells before execution.

In [4]:
import os
import pandas as pd
from datasets import load_dataset

SMOL_DATASET = "google/smol"
OUTPUT_DIR = "smol_csvs"
ROWS_PER_LANGUAGE = 200

os.makedirs(OUTPUT_DIR, exist_ok=True)

LOW_RESOURCE = [
    ("smolsent__en_am", "Amharic"),
    ("smolsent__en_sw", "Swahili"),
    ("smolsent__en_af", "Afrikaans"),
]

In [5]:
def load_first_n_rows(config_name: str, language_name: str, n: int = 200) -> pd.DataFrame:
    """
    Load the first n rows from a SMOL config and return them in the required format.

    Output columns:
      - human translation: English text
      - source text: target-language text
      - source language: readable language name
    """
    try:
        ds = load_dataset(SMOL_DATASET, config_name, split="train")
    except Exception as e:
        raise RuntimeError(f"Failed to load {config_name}: {e}")

    if len(ds) < n:
        raise ValueError(f"{config_name} only has {len(ds)} rows, but {n} were requested.")

    required_cols = {"src", "trg"}
    missing = required_cols - set(ds.column_names)
    if missing:
        raise ValueError(
            f"{config_name} is missing expected columns: {missing}. "
            f"Available columns: {ds.column_names}"
        )

    ds_small = ds.select(range(n))

    df = pd.DataFrame({
        "human translation": ds_small["src"],   # English text
        "source text": ds_small["trg"],         # Target-language text
        "source language": [language_name] * n,
    })

    # Remove empty/null rows just in case
    df = df.dropna(subset=["human translation", "source text"])
    df["human translation"] = df["human translation"].astype(str).str.strip()
    df["source text"] = df["source text"].astype(str).str.strip()
    df = df[
        (df["human translation"] != "") &
        (df["source text"] != "")
    ].reset_index(drop=True)

    if len(df) < n:
        raise ValueError(
            f"After cleaning, {config_name} has only {len(df)} valid rows; expected {n}."
        )

    return df.iloc[:n].copy()


def build_group_csv(group_configs, output_path: str, rows_per_language: int = 200) -> pd.DataFrame:
    """
    Build one combined CSV from a list of (config_name, language_name) pairs.
    Preserves order exactly as given in group_configs.
    """
    parts = []

    for config_name, language_name in group_configs:
        print(f"Loading {config_name} ({language_name}) ...")
        df_lang = load_first_n_rows(config_name, language_name, rows_per_language)
        parts.append(df_lang)
        print(f"  Added {len(df_lang)} rows")

    combined = pd.concat(parts, ignore_index=True)
    combined.to_csv(output_path, index=False, encoding="utf-8")
    print(f"\nSaved {len(combined)} rows to: {output_path}")

    # Quick sanity check
    print("\nCounts by source language:")
    print(combined["source language"].value_counts())

    return combined


In [ ]:
low_output = os.path.join(OUTPUT_DIR, "smol_test.csv")


low_df = build_group_csv(LOW_RESOURCE, low_output, ROWS_PER_LANGUAGE)

print("\nDone.")

In [1]:
import os
import pandas as pd


def extract_column_name(filename):
    """
    Extract column name from filename.
    Example: openwho_test_gemini.txt -> gemini
    """
    base = os.path.basename(filename)
    name = os.path.splitext(base)[0]  # remove .txt
    return name.split("_")[-1]        # take last part


def read_txt_file(filepath):
    """
    Read txt file and return cleaned lines.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines()]
    
    # Remove empty lines if any
    lines = [line for line in lines if line]
    return lines


def add_txt_columns_to_csv(csv_path, txt_folder, output_path):
    # Load CSV
    df = pd.read_csv(csv_path)
    num_rows = len(df)

    print(f"CSV loaded with {num_rows} rows")

    # Process each txt file
    for file in os.listdir(txt_folder):
        if file.endswith(".txt"):
            file_path = os.path.join(txt_folder, file)

            column_name = extract_column_name(file)
            lines = read_txt_file(file_path)

            print(f"Processing {file} → column '{column_name}'")

            # Validate length
            if len(lines) != num_rows:
                raise ValueError(
                    f"Row mismatch in {file}: "
                    f"{len(lines)} lines vs {num_rows} CSV rows"
                )

            # Add column
            df[column_name] = lines

    # Save new CSV
    df.to_csv(output_path, index=False)
    print(f"\nSaved updated CSV to: {output_path}")

In [ ]:
csv_file = "" # Set the path to generated CSV file above
txt_folder = " " # Set the path to the folder containing the txt files with machine translation outputs
output_file = " output.csv" # Set the path for the output CSV file with all translations

add_txt_columns_to_csv(csv_file, txt_folder, output_file)